In [2]:
import pandas as pd
df=pd.read_csv('results.csv')
df.shape
df.info()
df.describe()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 49477 entries, 0 to 49476
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   date        49477 non-null  object 
 1   home_team   49477 non-null  object 
 2   away_team   49477 non-null  object 
 3   home_score  49409 non-null  float64
 4   away_score  49409 non-null  float64
 5   tournament  49477 non-null  object 
 6   city        49477 non-null  object 
 7   country     49477 non-null  object 
 8   neutral     49477 non-null  bool   
dtypes: bool(1), float64(2), object(6)
memory usage: 3.1+ MB


,home_score,away_score
count,49409.000000,49409.000000
mean,1.757291,1.181890
std,1.774255,1.402027
min,0.000000,0.000000
25%,1.000000,0.000000
50%,1.000000,1.000000
75%,2.000000,2.000000
max,31.000000,21.000000


In [3]:
df['date'] = pd.to_datetime(df['date'])
# Séparer matchs joués vs à prédire
today = pd.Timestamp('2026-06-14')
# Matchs déjà joués (avec scores)
df_played = df[df['home_score'].notna() & (df['date'] <= today)].copy()

# Matchs FIFA 2026 Phase 1 à prédire (scores NA)
df_wc2026 = df[df['home_score'].isna() & 
               (df['tournament'] == 'FIFA World Cup')].copy()
print(f"Matchs joués pour entraînement : {len(df_played)}")
print(f"Matchs FIFA 2026 à prédire : {len(df_wc2026)}")
print(f"\nPériode entraînement : {df_played['date'].min()} → {df_played['date'].max()}")
print(f"\nMatchs à prédire :\n{df_wc2026[['date','home_team','away_team']].to_string()}")

Matchs joués pour entraînement : 49409
Matchs FIFA 2026 à prédire : 68

Période entraînement : 1872-11-30 00:00:00 → 2026-06-12 00:00:00

Matchs à prédire :
            date               home_team               away_team
49409 2026-06-13                   Qatar             Switzerland
49410 2026-06-13                  Brazil                 Morocco
49411 2026-06-13                   Haiti                Scotland
49412 2026-06-13               Australia                  Turkey
49413 2026-06-14                 Germany                 Curaçao
49414 2026-06-14             Ivory Coast                 Ecuador
49415 2026-06-14             Netherlands                   Japan
49416 2026-06-14                  Sweden                 Tunisia
49417 2026-06-15                 Belgium                   Egypt
49418 2026-06-15                    Iran             New Zealand
49419 2026-06-15                   Spain              Cape Verde
49420 2026-06-15            Saudi Arabia                 Urugua

In [4]:
# ============================================================
# ÉTAPE 2 — Filtrage + Feature Engineering
# ============================================================

# 1. Garder seulement depuis 2010
df_train = df_played[df_played['date'] >= '2010-01-01'].copy()
print(f"Matchs après 2010 : {len(df_train)}")

# 2. Poids par type de tournoi
tournament_weights = {
    'FIFA World Cup'                    : 5,
    'UEFA Euro'                         : 4,
    'Copa América'                      : 4,
    'Africa Cup of Nations'             : 4,
    'AFC Asian Cup'                     : 4,
    'CONCACAF Gold Cup'                 : 3,
    'FIFA World Cup qualification'      : 3,
    'UEFA Euro qualification'           : 3,
    'Copa América qualification'        : 3,
    'Africa Cup of Nations qualification': 3,
    'AFC Asian Cup qualification'       : 3,
    'Friendly'                          : 1,
}

def get_weight(tournament):
    for key in tournament_weights:
        if key in tournament:
            return tournament_weights[key]
    return 2  # autres compétitions officielles

df_train['weight'] = df_train['tournament'].apply(get_weight)

print("\nDistribution des poids :")
print(df_train['weight'].value_counts())

# ============================================================
# ÉTAPE 3 — Calcul des features par équipe
# ============================================================

def compute_team_features(team, date, df, n_matches=10):
    """
    Pour une équipe donnée, calcule ses stats sur les n derniers matchs
    avant une date donnée.
    """
    # Matchs où l'équipe a joué (domicile ou extérieur)
    mask = (
        ((df['home_team'] == team) | (df['away_team'] == team)) &
        (df['date'] < date)
    )
    matches = df[mask].sort_values('date', ascending=False).head(n_matches)

    if len(matches) == 0:
        return {
            'form_points'     : 0,
            'goals_scored_avg': 0,
            'goals_conceded_avg': 0,
            'win_rate'        : 0,
            'draw_rate'       : 0,
            'n_matches'       : 0,
        }

    points, goals_scored, goals_conceded = [], [], []

    for _, row in matches.iterrows():
        if row['home_team'] == team:
            gs, gc = row['home_score'], row['away_score']
        else:
            gs, gc = row['away_score'], row['home_score']

        goals_scored.append(gs)
        goals_conceded.append(gc)

        if gs > gc:
            points.append(3)
        elif gs == gc:
            points.append(1)
        else:
            points.append(0)

    return {
        'form_points'       : sum(points) / len(points),
        'goals_scored_avg'  : sum(goals_scored) / len(goals_scored),
        'goals_conceded_avg': sum(goals_conceded) / len(goals_conceded),
        'win_rate'          : points.count(3) / len(points),
        'draw_rate'         : points.count(1) / len(points),
        'n_matches'         : len(matches),
    }

def compute_h2h(home_team, away_team, date, df, n=5):
    """
    Head-to-head entre deux équipes sur les n derniers matchs.
    """
    mask = (
        (
            ((df['home_team'] == home_team) & (df['away_team'] == away_team)) |
            ((df['home_team'] == away_team) & (df['away_team'] == home_team))
        ) &
        (df['date'] < date)
    )
    matches = df[mask].sort_values('date', ascending=False).head(n)

    if len(matches) == 0:
        return {'h2h_home_wins': 0, 'h2h_away_wins': 0, 'h2h_draws': 0}

    home_wins = draws = away_wins = 0
    for _, row in matches.iterrows():
        if row['home_team'] == home_team:
            hs, as_ = row['home_score'], row['away_score']
        else:
            hs, as_ = row['away_score'], row['home_score']

        if hs > as_:   home_wins += 1
        elif hs == as_: draws    += 1
        else:           away_wins += 1

    total = len(matches)
    return {
        'h2h_home_wins': home_wins / total,
        'h2h_away_wins': away_wins / total,
        'h2h_draws'    : draws / total,
    }

# ============================================================
# ÉTAPE 4 — Construire le dataset final
# ============================================================

print("\nConstruction des features (patience...)") 

rows = []
for _, match in df_train.iterrows():
    home = match['home_team']
    away = match['away_team']
    date = match['date']

    hf = compute_team_features(home, date, df_train)
    af = compute_team_features(away, date, df_train)
    h2h = compute_h2h(home, away, date, df_train)

    row = {
        # Target
        'home_score' : match['home_score'],
        'away_score' : match['away_score'],
        # Features domicile
        'home_form'          : hf['form_points'],
        'home_goals_scored'  : hf['goals_scored_avg'],
        'home_goals_conceded': hf['goals_conceded_avg'],
        'home_win_rate'      : hf['win_rate'],
        # Features extérieur
        'away_form'          : af['form_points'],
        'away_goals_scored'  : af['goals_scored_avg'],
        'away_goals_conceded': af['goals_conceded_avg'],
        'away_win_rate'      : af['win_rate'],
        # H2H
        **h2h,
        # Contexte
        'neutral'  : int(match['neutral']),
        'weight'   : match['weight'],
    }
    rows.append(row)

df_features = pd.DataFrame(rows)
print(f"Dataset features : {df_features.shape}")
print(df_features.head())

Matchs après 2010 : 15821

Distribution des poids :
weight
2    5257
1    5073
5    3682
4    1809
Name: count, dtype: int64

Construction des features (patience...)
Dataset features : (15821, 15)
   home_score  away_score  home_form  home_goals_scored  home_goals_conceded  \
0         1.0         0.0        0.0                0.0                  0.0   
1         0.0         0.0        0.0                0.0                  0.0   
2         6.0         0.0        0.0                0.0                  0.0   
3         0.0         1.0        0.0                0.0                  0.0   
4         1.0         1.0        0.0                0.0                  0.0   

   home_win_rate  away_form  away_goals_scored  away_goals_conceded  \
0            0.0        0.0                0.0                  0.0   
1            0.0        0.0                0.0                  0.0   
2            0.0        0.0                0.0                  0.0   
3            0.0        0.0           

In [ ]:
# ============================================================
# ÉTAPE 5 — MODÈLE : Régression de Poisson
# ============================================================
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
import numpy as np

# Features utilisées
FEATURES = [
    'home_form', 'home_goals_scored', 'home_goals_conceded', 'home_win_rate',
    'away_form', 'away_goals_scored', 'away_goals_conceded', 'away_win_rate',
    'h2h_home_wins', 'h2h_away_wins', 'h2h_draws',
    'neutral', 'weight'
]

X = df_features[FEATURES]
y_home = df_features['home_score']
y_away = df_features['away_score']
w = df_features['weight']

# Split train/test
X_train, X_test, yh_train, yh_test, ya_train, ya_test, w_train, w_test = \
    train_test_split(X, y_home, y_away, w, test_size=0.2, random_state=42)

# Modèle buts domicile
model_home = GradientBoostingRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=4,
    loss='squared_error',
    random_state=42
)
model_home.fit(X_train, yh_train, sample_weight=w_train)

# Modèle buts extérieur
model_away = GradientBoostingRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=4,
    loss='squared_error',
    random_state=42
)
model_away.fit(X_train, ya_train, sample_weight=w_train)

# Évaluation
pred_home = model_home.predict(X_test)
pred_away = model_away.predict(X_test)

print("=== ÉVALUATION SUR TEST SET ===")
print(f"MAE buts domicile : {mean_absolute_error(yh_test, pred_home):.3f}")
print(f"MAE buts extérieur: {mean_absolute_error(ya_test, pred_away):.3f}")

# Vérification résultats prédits vs réels
def score_to_result(h, a):
    if h > a: return 'H'
    elif h == a: return 'D'
    else: return 'A'

real_results = [score_to_result(h, a) for h, a in zip(yh_test, ya_test)]
pred_results = [score_to_result(h, a) for h, a in zip(pred_home, pred_away)]

accuracy = sum(r == p for r, p in zip(real_results, pred_results)) / len(real_results)
print(f"Accuracy résultat (1/X/2) : {accuracy:.3f}")

# ============================================================
# ÉTAPE 6 — PRÉDICTION des 68 matchs FIFA 2026
# ============================================================

print("\n=== CONSTRUCTION FEATURES POUR LES 68 MATCHS ===")

rows_wc = []
for _, match in df_wc2026.iterrows():
    home = match['home_team']
    away = match['away_team']
    date = match['date']

    # On utilise tout df_played comme historique (pas seulement df_train)
    hf  = compute_team_features(home, date, df_played)
    af  = compute_team_features(away, date, df_played)
    h2h = compute_h2h(home, away, date, df_played)

    rows_wc.append({
        'date'     : date,
        'home_team': home,
        'away_team': away,
        'home_form'          : hf['form_points'],
        'home_goals_scored'  : hf['goals_scored_avg'],
        'home_goals_conceded': hf['goals_conceded_avg'],
        'home_win_rate'      : hf['win_rate'],
        'away_form'          : af['form_points'],
        'away_goals_scored'  : af['goals_scored_avg'],
        'away_goals_conceded': af['goals_conceded_avg'],
        'away_win_rate'      : af['win_rate'],
        **h2h,
        'neutral': int(match['neutral']),
        'weight' : 5,  # FIFA World Cup
    })

df_wc_features = pd.DataFrame(rows_wc)

# Prédiction
X_wc = df_wc_features[FEATURES]
df_wc_features['pred_home'] = np.maximum(0, model_home.predict(X_wc))
df_wc_features['pred_away'] = np.maximum(0, model_away.predict(X_wc))

# Arrondir pour avoir des scores entiers
df_wc_features['pred_home_r'] = df_wc_features['pred_home'].round().astype(int)
df_wc_features['pred_away_r'] = df_wc_features['pred_away'].round().astype(int)

print("\n=== SCORES PRÉDITS ===")
print(df_wc_features[['date','home_team','away_team',
                        'pred_home_r','pred_away_r']].to_string(index=False))

=== ÉVALUATION SUR TEST SET ===
MAE buts domicile : 1.087
MAE buts extérieur: 0.902
Accuracy résultat (1/X/2) : 0.548

=== CONSTRUCTION FEATURES POUR LES 68 MATCHS ===
Accuracy avec seuil nul : 0.500


In [ ]:
import os
import requests
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv("API_KEY")

headers = {
    "X-Auth-Token": API_KEY
}

url = "https://api.football-data.org/v4/competitions/WC/matches"

response = requests.get(url, headers=headers)
data = response.json()

groups = {}

for match in data["matches"]:
    group = match.get("group", "")

    if group and "GROUP" in group:
        home = match["homeTeam"]["name"]
        away = match["awayTeam"]["name"]

        if group not in groups:
            groups[group] = set()

        groups[group].add(home)
        groups[group].add(away)

for g, teams in sorted(groups.items()):
    print(f"{g}: {sorted(teams)}")

GROUP_A: ['Czechia', 'Mexico', 'South Africa', 'South Korea']
GROUP_B: ['Bosnia-Herzegovina', 'Canada', 'Qatar', 'Switzerland']
GROUP_C: ['Brazil', 'Haiti', 'Morocco', 'Scotland']
GROUP_D: ['Australia', 'Paraguay', 'Turkey', 'United States']
GROUP_E: ['Curaçao', 'Ecuador', 'Germany', 'Ivory Coast']
GROUP_F: ['Japan', 'Netherlands', 'Sweden', 'Tunisia']
GROUP_G: ['Belgium', 'Egypt', 'Iran', 'New Zealand']
GROUP_H: ['Cape Verde Islands', 'Saudi Arabia', 'Spain', 'Uruguay']
GROUP_I: ['France', 'Iraq', 'Norway', 'Senegal']
GROUP_J: ['Algeria', 'Argentina', 'Austria', 'Jordan']
GROUP_K: ['Colombia', 'Congo DR', 'Portugal', 'Uzbekistan']
GROUP_L: ['Croatia', 'England', 'Ghana', 'Panama']


In [7]:
# ============================================================
# ÉTAPE 7 — CLASSEMENT PAR GROUPE
# ============================================================

# Correction des noms (API vs notre dataset)
name_mapping = {
    'Czechia'           : 'Czech Republic',
    'Bosnia-Herzegovina': 'Bosnia and Herzegovina',
    'Congo DR'          : 'DR Congo',
    'Cape Verde Islands': 'Cape Verde',
}

groups_official = {
    'A': ['Mexico', 'South Korea', 'Czech Republic', 'South Africa'],
    'B': ['Canada', 'Qatar', 'Switzerland', 'Bosnia and Herzegovina'],
    'C': ['Brazil', 'Morocco', 'Scotland', 'Haiti'],
    'D': ['United States', 'Australia', 'Turkey', 'Paraguay'],
    'E': ['Germany', 'Ivory Coast', 'Ecuador', 'Curaçao'],
    'F': ['Netherlands', 'Japan', 'Sweden', 'Tunisia'],
    'G': ['Belgium', 'Iran', 'New Zealand', 'Egypt'],
    'H': ['Spain', 'Saudi Arabia', 'Uruguay', 'Cape Verde'],
    'I': ['France', 'Norway', 'Senegal', 'Iraq'],
    'J': ['Argentina', 'Algeria', 'Austria', 'Jordan'],
    'K': ['Portugal', 'Uzbekistan', 'Colombia', 'DR Congo'],
    'L': ['England', 'Croatia', 'Ghana', 'Panama'],
}

def calculate_standings(group_name, teams, predictions):
    """Calcule le classement d'un groupe"""
    
    # Initialiser stats
    stats = {t: {'pts':0, 'j':0, 'g':0, 'n':0, 'p':0, 'gf':0, 'ga':0, 'gd':0} 
             for t in teams}
    
    # Trouver les matchs du groupe dans les prédictions
    group_matches = predictions[
        predictions['home_team'].isin(teams) & 
        predictions['away_team'].isin(teams)
    ]
    
    for _, match in group_matches.iterrows():
        home = match['home_team']
        away = match['away_team']
        hg   = match['pred_home_r']
        ag   = match['pred_away_r']
        
        if home not in stats or away not in stats:
            continue
            
        # Buts
        stats[home]['gf'] += hg
        stats[home]['ga'] += ag
        stats[away]['gf'] += ag
        stats[away]['ga'] += hg
        stats[home]['j']  += 1
        stats[away]['j']  += 1
        
        # Points
        if hg > ag:
            stats[home]['pts'] += 3
            stats[home]['g']   += 1
            stats[away]['p']   += 1
        elif hg == ag:
            stats[home]['pts'] += 1
            stats[away]['pts'] += 1
            stats[home]['n']   += 1
            stats[away]['n']   += 1
        else:
            stats[away]['pts'] += 3
            stats[away]['g']   += 1
            stats[home]['p']   += 1
    
    # Différence de buts
    for t in teams:
        stats[t]['gd'] = stats[t]['gf'] - stats[t]['ga']
    
    # Classement (pts > gd > gf)
    ranked = sorted(teams, key=lambda t: (
        stats[t]['pts'], stats[t]['gd'], stats[t]['gf']
    ), reverse=True)
    
    return ranked, stats

# ============================================================
# Calculer et afficher tous les groupes
# ============================================================

print("=" * 65)
print("CLASSEMENT PHASE DE GROUPES — FIFA WORLD CUP 2026 (PRÉDIT)")
print("=" * 65)

all_group_results = {}

for group_name, teams in groups_official.items():
    ranked, stats = calculate_standings(group_name, teams, df_wc_features)
    all_group_results[group_name] = {'ranked': ranked, 'stats': stats}
    
    print(f"\n{'─'*65}")
    print(f"  GROUPE {group_name}")
    print(f"{'─'*65}")
    print(f"  {'Équipe':<25} {'J':>2} {'G':>2} {'N':>2} {'P':>2} {'GF':>3} {'GA':>3} {'GD':>4} {'Pts':>4}")
    print(f"  {'─'*57}")
    
    for i, team in enumerate(ranked):
        s = stats[team]
        qualifier = " ✅" if i < 2 else ""
        print(f"  {i+1}. {team:<22} {s['j']:>2} {s['g']:>2} {s['n']:>2} {s['p']:>2} "
              f"{s['gf']:>3} {s['ga']:>3} {s['gd']:>4} {s['pts']:>4}{qualifier}")

# ============================================================
# Résumé des qualifiés
# ============================================================
print(f"\n{'='*65}")
print("ÉQUIPES QUALIFIÉES POUR LES 8èmes DE FINALE (PRÉDIT)")
print(f"{'='*65}")

qualified = []
third_place = []

for group_name, result in all_group_results.items():
    ranked = result['ranked']
    stats  = result['stats']
    q1, q2 = ranked[0], ranked[1]
    q3     = ranked[2]
    qualified.append((group_name, '1er', q1, stats[q1]['pts'], stats[q1]['gd']))
    qualified.append((group_name, '2e',  q2, stats[q2]['pts'], stats[q2]['gd']))
    third_place.append((group_name, q3, stats[q3]['pts'], stats[q3]['gd']))

print(f"\n{'Groupe':<8} {'Place':<5} {'Équipe':<25} {'Pts':>4} {'GD':>4}")
print(f"{'─'*50}")
for g, place, team, pts, gd in qualified:
    print(f"  {g:<6} {place:<5} {team:<25} {pts:>4} {gd:>4}")

# Meilleurs 3es (8 groupes de 3es se qualifient sur 12)
print(f"\n{'─'*65}")
print("MEILLEURS 3es (les 8 meilleurs sur 12 se qualifient)")
print(f"{'─'*65}")
third_sorted = sorted(third_place, key=lambda x: (x[2], x[3]), reverse=True)
for i, (g, team, pts, gd) in enumerate(third_sorted):
    qualify = " ✅" if i < 8 else " ❌"
    print(f"  Groupe {g} — {team:<25} Pts:{pts:>2} GD:{gd:>3}{qualify}")

CLASSEMENT PHASE DE GROUPES — FIFA WORLD CUP 2026 (PRÉDIT)

─────────────────────────────────────────────────────────────────
  GROUPE A
─────────────────────────────────────────────────────────────────
  Équipe                     J  G  N  P  GF  GA   GD  Pts
  ─────────────────────────────────────────────────────────
  1. Mexico                  2  2  0  0   4   2    2    6 ✅
  2. Czech Republic          2  1  0  1   3   3    0    3 ✅
  3. South Korea             2  0  1  1   2   3   -1    1
  4. South Africa            2  0  1  1   2   3   -1    1

─────────────────────────────────────────────────────────────────
  GROUPE B
─────────────────────────────────────────────────────────────────
  Équipe                     J  G  N  P  GF  GA   GD  Pts
  ─────────────────────────────────────────────────────────
  1. Switzerland             3  1  2  0   4   3    1    5 ✅
  2. Canada                  2  1  1  0   3   2    1    4 ✅
  3. Bosnia and Herzegovina  2  0  2  0   2   2    0    2
  4

In [8]:
# Compter les matchs par équipe dans df_wc_features
from collections import defaultdict

match_count = defaultdict(int)
for _, row in df_wc_features.iterrows():
    match_count[row['home_team']] += 1
    match_count[row['away_team']] += 1

print("Matchs par équipe :")
for team, count in sorted(match_count.items()):
    flag = " ⚠️" if count != 3 else ""
    print(f"  {team:<30} {count} matchs{flag}")

print(f"\nTotal matchs dans df_wc_features : {len(df_wc_features)}")
print(f"Attendu : 48 matchs (chaque groupe = 6 matchs x 12 groupes)")

# Vérifier les matchs du groupe A spécifiquement
print("\n--- Matchs Groupe A ---")
group_a = ['Mexico', 'South Korea', 'Czech Republic', 'South Africa']
for _, row in df_wc_features.iterrows():
    if row['home_team'] in group_a and row['away_team'] in group_a:
        print(f"  {row['home_team']} vs {row['away_team']} → {row['pred_home_r']}-{row['pred_away_r']}")

Matchs par équipe :
  Algeria                        3 matchs
  Argentina                      3 matchs
  Australia                      3 matchs
  Austria                        3 matchs
  Belgium                        3 matchs
  Bosnia and Herzegovina         2 matchs ⚠️
  Brazil                         3 matchs
  Canada                         2 matchs ⚠️
  Cape Verde                     3 matchs
  Colombia                       3 matchs
  Croatia                        3 matchs
  Curaçao                        3 matchs
  Czech Republic                 2 matchs ⚠️
  DR Congo                       3 matchs
  Ecuador                        3 matchs
  Egypt                          3 matchs
  England                        3 matchs
  France                         3 matchs
  Germany                        3 matchs
  Ghana                          3 matchs
  Haiti                          3 matchs
  Iran                           3 matchs
  Iraq                           3 matchs
  Ivo

In [11]:
# ============================================================
# FIX — Séparer matchs joués vs à prédire correctement
# ============================================================

# Matchs FIFA 2026 déjà joués (scores réels disponibles)
wc_played = df_played[
    (df_played['tournament'] == 'FIFA World Cup') & 
    (df_played['date'] >= '2026-06-11')
].copy()

print("Matchs FIFA 2026 déjà joués :")
print(wc_played[['date','home_team','away_team','home_score','away_score']].to_string())

# Matchs encore à jouer (nos prédictions)
wc_to_predict = df_wc_features.copy()

# Retirer de nos prédictions les matchs déjà joués
played_pairs = set(zip(wc_played['home_team'], wc_played['away_team']))
mask = wc_to_predict.apply(
    lambda r: (r['home_team'], r['away_team']) not in played_pairs, axis=1
)
wc_to_predict = wc_to_predict[mask].copy()

print(f"\nMatchs déjà joués : {len(wc_played)}")
print(f"Matchs à prédire  : {len(wc_to_predict)}")
print(f"Total             : {len(wc_played) + len(wc_to_predict)}")

# ============================================================
# Combiner scores réels + prédictions
# ============================================================

# Scores réels
real_results = wc_played[['home_team','away_team','home_score','away_score']].copy()
real_results.columns = ['home_team','away_team','pred_home_r','pred_away_r']
real_results['pred_home_r'] = real_results['pred_home_r'].astype(int)
real_results['pred_away_r'] = real_results['pred_away_r'].astype(int)
real_results['source'] = 'réel'

# Prédictions
pred_results = wc_to_predict[['home_team','away_team','pred_home_r','pred_away_r']].copy()
pred_results['source'] = 'prédit'

# Dataset complet
df_all_wc = pd.concat([real_results, pred_results], ignore_index=True)

print(f"\nDataset complet : {len(df_all_wc)} matchs")
print(f"  Réels  : {len(real_results)}")
print(f"  Prédits: {len(pred_results)}")

# Vérification par équipe
from collections import defaultdict
match_count = defaultdict(int)
for _, row in df_all_wc.iterrows():
    match_count[row['home_team']] += 1
    match_count[row['away_team']] += 1

problems = [(t, c) for t, c in match_count.items() if c != 3]
if problems:
    print("\n⚠️ Équipes avec ≠ 3 matchs :")
    for t, c in problems:
        print(f"  {t}: {c}")
else:
    print("\n✅ Toutes les équipes ont exactement 3 matchs !")

Matchs FIFA 2026 déjà joués :
            date      home_team               away_team  home_score  away_score
49405 2026-06-11         Mexico            South Africa         2.0         0.0
49406 2026-06-11    South Korea          Czech Republic         2.0         1.0
49407 2026-06-12         Canada  Bosnia and Herzegovina         1.0         1.0
49408 2026-06-12  United States                Paraguay         4.0         1.0

Matchs déjà joués : 4
Matchs à prédire  : 68
Total             : 72

Dataset complet : 72 matchs
  Réels  : 4
  Prédits: 68

✅ Toutes les équipes ont exactement 3 matchs !


In [12]:
# ============================================================
# FIX — Séparer matchs joués vs à prédire correctement
# ============================================================

# Matchs FIFA 2026 déjà joués (scores réels disponibles)
wc_played = df_played[
    (df_played['tournament'] == 'FIFA World Cup') & 
    (df_played['date'] >= '2026-06-11')
].copy()

print("Matchs FIFA 2026 déjà joués :")
print(wc_played[['date','home_team','away_team','home_score','away_score']].to_string())

# Matchs encore à jouer (nos prédictions)
wc_to_predict = df_wc_features.copy()

# Retirer de nos prédictions les matchs déjà joués
played_pairs = set(zip(wc_played['home_team'], wc_played['away_team']))
mask = wc_to_predict.apply(
    lambda r: (r['home_team'], r['away_team']) not in played_pairs, axis=1
)
wc_to_predict = wc_to_predict[mask].copy()

print(f"\nMatchs déjà joués : {len(wc_played)}")
print(f"Matchs à prédire  : {len(wc_to_predict)}")
print(f"Total             : {len(wc_played) + len(wc_to_predict)}")

# ============================================================
# Combiner scores réels + prédictions
# ============================================================

# Scores réels
real_results = wc_played[['home_team','away_team','home_score','away_score']].copy()
real_results.columns = ['home_team','away_team','pred_home_r','pred_away_r']
real_results['pred_home_r'] = real_results['pred_home_r'].astype(int)
real_results['pred_away_r'] = real_results['pred_away_r'].astype(int)
real_results['source'] = 'réel'

# Prédictions
pred_results = wc_to_predict[['home_team','away_team','pred_home_r','pred_away_r']].copy()
pred_results['source'] = 'prédit'

# Dataset complet
df_all_wc = pd.concat([real_results, pred_results], ignore_index=True)

print(f"\nDataset complet : {len(df_all_wc)} matchs")
print(f"  Réels  : {len(real_results)}")
print(f"  Prédits: {len(pred_results)}")

# Vérification par équipe
from collections import defaultdict
match_count = defaultdict(int)
for _, row in df_all_wc.iterrows():
    match_count[row['home_team']] += 1
    match_count[row['away_team']] += 1

problems = [(t, c) for t, c in match_count.items() if c != 3]
if problems:
    print("\n⚠️ Équipes avec ≠ 3 matchs :")
    for t, c in problems:
        print(f"  {t}: {c}")
else:
    print("\n✅ Toutes les équipes ont exactement 3 matchs !")

Matchs FIFA 2026 déjà joués :
            date      home_team               away_team  home_score  away_score
49405 2026-06-11         Mexico            South Africa         2.0         0.0
49406 2026-06-11    South Korea          Czech Republic         2.0         1.0
49407 2026-06-12         Canada  Bosnia and Herzegovina         1.0         1.0
49408 2026-06-12  United States                Paraguay         4.0         1.0

Matchs déjà joués : 4
Matchs à prédire  : 68
Total             : 72

Dataset complet : 72 matchs
  Réels  : 4
  Prédits: 68

✅ Toutes les équipes ont exactement 3 matchs !


In [13]:
# ============================================================
# FIX DÉFINITIF
# ============================================================

# Les 4 matchs joués (home_team, away_team)
played_pairs = {
    ('Mexico', 'South Africa'),
    ('South Korea', 'Czech Republic'),
    ('Canada', 'Bosnia and Herzegovina'),
    ('United States', 'Paraguay'),
}

# Reconstruire df_wc_features depuis zéro en excluant ces paires
rows_wc = []
for _, match in df_wc2026.iterrows():
    home = match['home_team']
    away = match['away_team']
    
    # Sauter si déjà joué
    if (home, away) in played_pairs:
        continue
    
    date = match['date']
    hf  = compute_team_features(home, date, df_played)
    af  = compute_team_features(away, date, df_played)
    h2h = compute_h2h(home, away, date, df_played)

    rows_wc.append({
        'home_team': home, 'away_team': away, 'date': date,
        'home_form'          : hf['form_points'],
        'home_goals_scored'  : hf['goals_scored_avg'],
        'home_goals_conceded': hf['goals_conceded_avg'],
        'home_win_rate'      : hf['win_rate'],
        'away_form'          : af['form_points'],
        'away_goals_scored'  : af['goals_scored_avg'],
        'away_goals_conceded': af['goals_conceded_avg'],
        'away_win_rate'      : af['win_rate'],
        **h2h,
        'neutral': int(match['neutral']),
        'weight' : 5,
    })

wc_to_predict = pd.DataFrame(rows_wc)

# Prédire
X_wc2 = wc_to_predict[FEATURES]
wc_to_predict['pred_home_r'] = np.maximum(0, model_home.predict(X_wc2)).round().astype(int)
wc_to_predict['pred_away_r'] = np.maximum(0, model_away.predict(X_wc2)).round().astype(int)
wc_to_predict['source'] = 'prédit'

print(f"Matchs à prédire : {len(wc_to_predict)}")  # doit être 64

# Scores réels
real_results = wc_played[['home_team','away_team','home_score','away_score']].copy()
real_results.columns = ['home_team','away_team','pred_home_r','pred_away_r']
real_results['pred_home_r'] = real_results['pred_home_r'].astype(int)
real_results['pred_away_r'] = real_results['pred_away_r'].astype(int)
real_results['source'] = 'réel'

# Combiner
df_all_wc = pd.concat([
    real_results[['home_team','away_team','pred_home_r','pred_away_r','source']],
    wc_to_predict[['home_team','away_team','pred_home_r','pred_away_r','source']]
], ignore_index=True)

print(f"Total matchs : {len(df_all_wc)}")  # doit être 68

# Vérification
match_count = defaultdict(int)
for _, row in df_all_wc.iterrows():
    match_count[row['home_team']] += 1
    match_count[row['away_team']] += 1

problems = [(t,c) for t,c in match_count.items() if c != 3]
if problems:
    print("⚠️ Problèmes :", problems)
else:
    print("✅ Toutes les équipes ont exactement 3 matchs !")

# Vérif Groupe A
print("\nGroupe A :")
group_a = ['Mexico', 'South Korea', 'Czech Republic', 'South Africa']
for _, row in df_all_wc.iterrows():
    if row['home_team'] in group_a and row['away_team'] in group_a:
        print(f"  {row['home_team']} vs {row['away_team']} "
              f"→ {row['pred_home_r']}-{row['pred_away_r']} [{row['source']}]")

Matchs à prédire : 68
Total matchs : 72
✅ Toutes les équipes ont exactement 3 matchs !

Groupe A :
  Mexico vs South Africa → 2-0 [réel]
  South Korea vs Czech Republic → 2-1 [réel]
  Czech Republic vs South Africa → 2-1 [prédit]
  Mexico vs South Korea → 2-1 [prédit]
  Mexico vs Czech Republic → 2-1 [prédit]
  South Africa vs South Korea → 1-1 [prédit]


In [ ]:
# Redéfinir df_wc2026 correctement — seulement les vrais matchs futurs
df_wc2026_clean = df[
    (df['home_score'].isna()) & 
    (df['tournament'] == 'FIFA World Cup') &
    (df['date'] > pd.Timestamp('2026-06-12'))  # après les matchs déjà joués
].copy()

print(f"df_wc2026_clean : {len(df_wc2026_clean)} matchs")
print(df_wc2026_clean['date'].value_counts().sort_index())

df_wc2026_clean : 68 matchs
date
2026-06-13    4
2026-06-14    4
2026-06-15    4
2026-06-16    4
2026-06-17    4
2026-06-18    4
2026-06-19    4
2026-06-20    4
2026-06-21    4
2026-06-22    4
2026-06-23    4
2026-06-24    6
2026-06-25    6
2026-06-26    6
2026-06-27    6
Name: count, dtype: int64


In [14]:
# ============================================================
# FIX FINAL — Reconstruire proprement les 68 matchs (4+64)
# ============================================================

# Tous les matchs FIFA 2026 Phase 1 = joués + à prédire
all_wc_matches = pd.concat([
    # Matchs joués (scores réels)
    wc_played[['home_team','away_team','home_score','away_score']].rename(
        columns={'home_score':'pred_home_r','away_score':'pred_away_r'}
    ).assign(source='réel'),
    # Matchs prédits
    wc_to_predict[['home_team','away_team','pred_home_r','pred_away_r']].assign(source='prédit')
], ignore_index=True)

# Vérification par groupe
print("Matchs par groupe :")
for group_name, teams in groups_official.items():
    count = len(all_wc_matches[
        all_wc_matches['home_team'].isin(teams) & 
        all_wc_matches['away_team'].isin(teams)
    ])
    flag = "✅" if count == 6 else f"⚠️ ({count})"
    print(f"  Groupe {group_name}: {count} matchs {flag}")

print(f"\nTotal : {len(all_wc_matches)} matchs")

# Vérification matchs par équipe
match_count = defaultdict(int)
for _, row in all_wc_matches.iterrows():
    match_count[row['home_team']] += 1
    match_count[row['away_team']] += 1

problems = [(t,c) for t,c in match_count.items() if c != 3]
if problems:
    print("⚠️ Équipes ≠ 3 matchs :", problems)
else:
    print("✅ Toutes les équipes ont exactement 3 matchs !")

# Groupe A détail
print("\nGroupe A détail :")
group_a = groups_official['A']
for _, row in all_wc_matches.iterrows():
    if row['home_team'] in group_a and row['away_team'] in group_a:
        print(f"  {row['home_team']} vs {row['away_team']} "
              f"→ {row['pred_home_r']}-{row['pred_away_r']} [{row['source']}]")

Matchs par groupe :
  Groupe A: 6 matchs ✅
  Groupe B: 6 matchs ✅
  Groupe C: 6 matchs ✅
  Groupe D: 6 matchs ✅
  Groupe E: 6 matchs ✅
  Groupe F: 6 matchs ✅
  Groupe G: 6 matchs ✅
  Groupe H: 6 matchs ✅
  Groupe I: 6 matchs ✅
  Groupe J: 6 matchs ✅
  Groupe K: 6 matchs ✅
  Groupe L: 6 matchs ✅

Total : 72 matchs
✅ Toutes les équipes ont exactement 3 matchs !

Groupe A détail :
  Mexico vs South Africa → 2.0-0.0 [réel]
  South Korea vs Czech Republic → 2.0-1.0 [réel]
  Czech Republic vs South Africa → 2.0-1.0 [prédit]
  Mexico vs South Korea → 2.0-1.0 [prédit]
  Mexico vs Czech Republic → 2.0-1.0 [prédit]
  South Africa vs South Korea → 1.0-1.0 [prédit]


In [15]:
# Convertir les scores en int
all_wc_matches['pred_home_r'] = all_wc_matches['pred_home_r'].astype(int)
all_wc_matches['pred_away_r'] = all_wc_matches['pred_away_r'].astype(int)

# ============================================================
# CLASSEMENT FINAL
# ============================================================

print("=" * 65)
print("CLASSEMENT PHASE DE GROUPES — FIFA WORLD CUP 2026 (PRÉDIT)")
print("=" * 65)

all_group_results = {}

for group_name, teams in groups_official.items():
    ranked, stats = calculate_standings(group_name, teams, all_wc_matches)
    all_group_results[group_name] = {'ranked': ranked, 'stats': stats}
    
    print(f"\n{'─'*65}")
    print(f"  GROUPE {group_name}")
    print(f"{'─'*65}")
    print(f"  {'Équipe':<25} {'J':>2} {'G':>2} {'N':>2} {'P':>2} {'GF':>3} {'GA':>3} {'GD':>4} {'Pts':>4}")
    print(f"  {'─'*57}")
    
    for i, team in enumerate(ranked):
        s = stats[team]
        qualifier = " ✅" if i < 2 else ""
        print(f"  {i+1}. {team:<22} {s['j']:>2} {s['g']:>2} {s['n']:>2} {s['p']:>2} "
              f"{s['gf']:>3} {s['ga']:>3} {s['gd']:>4} {s['pts']:>4}{qualifier}")

# Qualifiés + meilleurs 3es
print(f"\n{'='*65}")
print("ÉQUIPES QUALIFIÉES POUR LES 8èmes (PRÉDIT)")
print(f"{'='*65}")

qualified = []
third_place = []

for group_name, result in all_group_results.items():
    ranked = result['ranked']
    stats  = result['stats']
    qualified.append((group_name, '1er', ranked[0], stats[ranked[0]]['pts'], stats[ranked[0]]['gd']))
    qualified.append((group_name, '2e',  ranked[1], stats[ranked[1]]['pts'], stats[ranked[1]]['gd']))
    third_place.append((group_name, ranked[2], stats[ranked[2]]['pts'], stats[ranked[2]]['gd']))

print(f"\n{'Groupe':<8} {'Place':<5} {'Équipe':<25} {'Pts':>4} {'GD':>4}")
print(f"{'─'*50}")
for g, place, team, pts, gd in qualified:
    print(f"  {g:<6} {place:<5} {team:<25} {pts:>4} {gd:>4}")

print(f"\n{'─'*65}")
print("MEILLEURS 3es (8 meilleurs sur 12 se qualifient)")
print(f"{'─'*65}")
third_sorted = sorted(third_place, key=lambda x: (x[2], x[3]), reverse=True)
for i, (g, team, pts, gd) in enumerate(third_sorted):
    qualify = " ✅" if i < 8 else " ❌"
    print(f"  Groupe {g} — {team:<25} Pts:{pts:>2} GD:{gd:>3}{qualify}")

CLASSEMENT PHASE DE GROUPES — FIFA WORLD CUP 2026 (PRÉDIT)

─────────────────────────────────────────────────────────────────
  GROUPE A
─────────────────────────────────────────────────────────────────
  Équipe                     J  G  N  P  GF  GA   GD  Pts
  ─────────────────────────────────────────────────────────
  1. Mexico                  3  3  0  0   6   2    4    9 ✅
  2. South Korea             3  1  1  1   4   4    0    4 ✅
  3. Czech Republic          3  1  0  2   4   5   -1    3
  4. South Africa            3  0  1  2   2   5   -3    1

─────────────────────────────────────────────────────────────────
  GROUPE B
─────────────────────────────────────────────────────────────────
  Équipe                     J  G  N  P  GF  GA   GD  Pts
  ─────────────────────────────────────────────────────────
  1. Canada                  3  1  2  0   4   3    1    5 ✅
  2. Switzerland             3  1  2  0   4   3    1    5 ✅
  3. Bosnia and Herzegovina  3  0  3  0   3   3    0    3
  4

In [ ]:
# ============================================================
# AFFICHAGE DÉTAILLÉ — TOUS LES MATCHS PAR GROUPE
# ============================================================

print("=" * 65)
print("SCORES DÉTAILLÉS PAR GROUPE — FIFA WORLD CUP 2026")
print("=" * 65)

for group_name, teams in groups_official.items():
    print(f"\n{'─'*65}")
    print(f"  GROUPE {group_name} : {' | '.join(teams)}")
    print(f"{'─'*65}")
    
    group_matches = all_wc_matches[
        all_wc_matches['home_team'].isin(teams) & 
        all_wc_matches['away_team'].isin(teams)
    ].copy()
    
    for _, row in group_matches.iterrows():
        home = row['home_team']
        away = row['away_team']
        hg   = row['pred_home_r']
        ag   = row['pred_away_r']
        src  = row['source']
        
        if hg > ag:
            result = f"✅ {home}"
        elif hg == ag:
            result = "🤝 Nul"
        else:
            result = f"✅ {away}"
            
        tag = "🔴 RÉEL  " if src == 'réel' else "🔵 PRÉDIT"
        
        print(f"  {tag} | {home:<25} {hg} - {ag}  {away:<25} → {result}")
    
    # Classement du groupe
    ranked, stats = calculate_standings(group_name, teams, all_wc_matches)
    print(f"\n  Classement :")
    for i, team in enumerate(ranked):
        s = stats[team]
        q = "✅" if i < 2 else "  "
        print(f"  {q} {i+1}. {team:<25} {s['pts']} pts  GD:{s['gd']:+d}  ({s['gf']}:{s['ga']})")

SCORES DÉTAILLÉS PAR GROUPE — FIFA WORLD CUP 2026

─────────────────────────────────────────────────────────────────
  GROUPE A : Mexico | South Korea | Czech Republic | South Africa
─────────────────────────────────────────────────────────────────
  🔴 RÉEL   | Mexico                    2 - 0  South Africa              → ✅ Mexico
  🔴 RÉEL   | South Korea               2 - 1  Czech Republic            → ✅ South Korea
  🔵 PRÉDIT | Czech Republic            2 - 1  South Africa              → ✅ Czech Republic
  🔵 PRÉDIT | Mexico                    2 - 1  South Korea               → ✅ Mexico
  🔵 PRÉDIT | Mexico                    2 - 1  Czech Republic            → ✅ Mexico
  🔵 PRÉDIT | South Africa              1 - 1  South Korea               → 🤝 Nul

  Classement :
  ✅ 1. Mexico                    9 pts  GD:+4  (6:2)
  ✅ 2. South Korea               4 pts  GD:+0  (4:4)
     3. Czech Republic            3 pts  GD:-1  (4:5)
     4. South Africa              1 pts  GD:-3  (2:5)

────────────

In [16]:
# ============================================================
# MÉTRIQUES COMPLÈTES DU MODÈLE
# ============================================================
from sklearn.metrics import (
    mean_absolute_error, 
    mean_squared_error, 
    r2_score
)
import numpy as np

pred_home = model_home.predict(X_test)
pred_away = model_away.predict(X_test)

# ── 1. Métriques de régression ──────────────────────────────
mae_h  = mean_absolute_error(yh_test, pred_home)
mae_a  = mean_absolute_error(ya_test, pred_away)
rmse_h = np.sqrt(mean_squared_error(yh_test, pred_home))
rmse_a = np.sqrt(mean_squared_error(ya_test, pred_away))
r2_h   = r2_score(yh_test, pred_home)
r2_a   = r2_score(ya_test, pred_away)

print("=" * 50)
print("  MÉTRIQUES DE RÉGRESSION (buts)")
print("=" * 50)
print(f"{'Métrique':<20} {'Domicile':>10} {'Extérieur':>10}")
print("-" * 50)
print(f"{'MAE':<20} {mae_h:>10.3f} {mae_a:>10.3f}")
print(f"{'RMSE':<20} {rmse_h:>10.3f} {rmse_a:>10.3f}")
print(f"{'R²':<20} {r2_h:>10.3f} {r2_a:>10.3f}")

# ── 2. Accuracy résultat (1/X/2) ────────────────────────────
def score_to_result(h, a):
    if h > a:   return 'H'
    elif h == a: return 'D'
    else:        return 'A'

real_res = [score_to_result(h, a) for h, a in zip(yh_test, ya_test)]
pred_res = [score_to_result(h, a) for h, a in zip(pred_home, pred_away)]

accuracy = sum(r == p for r, p in zip(real_res, pred_res)) / len(real_res)

# Accuracy par type de résultat
from collections import Counter
real_counts = Counter(real_res)
correct_by_type = {'H': 0, 'D': 0, 'A': 0}
total_by_type   = {'H': 0, 'D': 0, 'A': 0}

for r, p in zip(real_res, pred_res):
    total_by_type[r] += 1
    if r == p:
        correct_by_type[r] += 1

print(f"\n{'=' * 50}")
print("  ACCURACY RÉSULTAT (1 / X / 2)")
print("=" * 50)
print(f"  Accuracy globale     : {accuracy:.3f} ({accuracy*100:.1f}%)")
print(f"\n  Détail par résultat :")
labels = {'H': 'Victoire domicile', 'D': 'Nul', 'A': 'Victoire extérieur'}
for key in ['H', 'D', 'A']:
    n     = total_by_type[key]
    ok    = correct_by_type[key]
    pct   = ok / n * 100 if n > 0 else 0
    print(f"    {labels[key]:<22} : {ok}/{n}  ({pct:.1f}%)")

# ── 3. Matrice de confusion ──────────────────────────────────
from sklearn.metrics import confusion_matrix, classification_report

print(f"\n{'=' * 50}")
print("  MATRICE DE CONFUSION")
print("=" * 50)
labels_order = ['H', 'D', 'A']
cm = confusion_matrix(real_res, pred_res, labels=labels_order)

print(f"\n  {'':>20} {'Prédit H':>10} {'Prédit D':>10} {'Prédit A':>10}")
print(f"  {'-'*52}")
for i, label in enumerate(labels_order):
    print(f"  {'Réel ' + labels[label]:<20} {cm[i][0]:>10} {cm[i][1]:>10} {cm[i][2]:>10}")

# ── 4. Importance des features ───────────────────────────────
print(f"\n{'=' * 50}")
print("  IMPORTANCE DES FEATURES")
print("=" * 50)
importances_h = model_home.feature_importances_
importances_a = model_away.feature_importances_
avg_imp = (importances_h + importances_a) / 2

feat_imp = sorted(zip(FEATURES, avg_imp), key=lambda x: x[1], reverse=True)
print(f"\n  {'Feature':<28} {'Importance':>10}")
print(f"  {'-'*40}")
for feat, imp in feat_imp:
    bar = "█" * int(imp * 100)
    print(f"  {feat:<28} {imp:>8.3f}  {bar}")

  MÉTRIQUES DE RÉGRESSION (buts)
Métrique               Domicile  Extérieur
--------------------------------------------------
MAE                       1.087      0.902
RMSE                      1.459      1.233
R²                        0.175      0.198

  ACCURACY RÉSULTAT (1 / X / 2)
  Accuracy globale     : 0.548 (54.8%)

  Détail par résultat :
    Victoire domicile      : 1220/1486  (82.1%)
    Nul                    : 0/746  (0.0%)
    Victoire extérieur     : 516/933  (55.3%)

  MATRICE DE CONFUSION

                         Prédit H   Prédit D   Prédit A
  ----------------------------------------------------
  Réel Victoire domicile       1220          0        266
  Réel Nul                    501          0        245
  Réel Victoire extérieur        417          0        516

  IMPORTANCE DES FEATURES

  Feature                      Importance
  ----------------------------------------
  home_goals_conceded             0.269  ██████████████████████████
  away_goals_concede